# Multi-hazard climate screening: Seville parkings

A physical-risk screening for a small portfolio of Seville car parks. The notebook first scores every hazard the platform models (chronic and acute) on a 0 to 3 scale, then explains where a specific damage function drives the result versus where the score comes straight from hazard intensity, and finally prices the two material hazards for this estate: flood (structure and vehicles; garage contents is on the way) and chronic heat (cooling demand and cost). A traceability section records which model was used for each hazard and why. Results are for screening, not regulatory submission.

**New to the Alpha-Klima API?** You may want to explore `asset_impact.ipynb` and `hazard_data.ipynb` first. The first builds an asset-impact request end to end and plots the binned impact distributions the service returns; the second does the same for point-level hazard intensity curves. Seeing those raw responses explains the shapes this notebook works with, and why the helpers below (`ratio_by_id`, `exceedance`, `distribution`, `score_matrix`, `heat_cdd_at_points`) are small extractors that pull one field out of a nested record rather than anything more elaborate.

**To run this notebook you need:**

- A `.env` file at the repository root with a valid Alpha-Klima API key and base URL: `ALPHA_KLIMA_API_KEY=...` and `ALPHA_KLIMA_API_BASE_URL=https://platform.alpha-klima.com/prapi`.
- A **portfolio of assets** in JSON (here `resources/parkings_seville.json`).
- The shared **helper modules** under `notebooks/auxiliary_functions/`: `geometry_maps_plots.py` (map), `financial_metrics_plots.py` (VaR / Expected Shortfall plots), `tail_metrics.py` (VaR / Expected Shortfall calculation, temporary) and `cooling_model.py` (cooling estimate).

All paths are relative to the notebook (`../.env`, `../resources/...`, `auxiliary_functions/...`). If you hit a `FileNotFoundError` or an import error, check these paths first and adjust them to your setup.

## 1. Setup and configuration

In [ ]:
import os
import json
import time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import requests
import plotly.graph_objects as go
from dotenv import load_dotenv
from IPython.display import display

from auxiliary_functions import geometry_maps_plots as gmp
from auxiliary_functions import financial_metrics_plots as fmp
from auxiliary_functions import cooling_model as cm
from auxiliary_functions import tail_metrics as tm

load_dotenv("../.env")
API_KEY = os.environ.get("ALPHA_KLIMA_API_KEY")
API_BASE = os.environ.get("ALPHA_KLIMA_API_BASE_URL", "").rstrip("/")
HEADERS = {"X-API-Key": API_KEY, "Accept": "application/json"}
assert API_KEY and API_BASE, "Set ALPHA_KLIMA_API_KEY and ALPHA_KLIMA_API_BASE_URL in ../.env"

`CONFIG` holds the whole run configuration in one place, so most of the exposure and scope assumptions are visible and editable here:

- **Portfolio**: which file to load (`parkings_portfolio`) and which assets to include (`parking_ids`).
- **Hazard scope and horizons**: the flood scenarios/years and the headline flood case; the heat scenario/year and the historical baseline; and `score_scope`, the one indicator per hazard used to build the 0 to 3 risk-score table, scored across `score_scenarios` and `score_years`.
- **Economic assumptions**: the electricity `tariff` (with source) for cooling cost, the `avg_vehicle_value` and `fleet_mix` behind vehicle exposure, and `m2_per_parking_space` for the floor-area-scaled cooling estimate.
- **Presentation**: the palette, score band colours and hazard display labels.

To retarget the analysis (different assets, scenarios or assumptions), edit `CONFIG` and re-run.

In [ ]:
CONFIG = {
    "parkings_portfolio": "../resources/parkings_seville.json", #assets under analysis
    "parking_ids": ["P1", "P2", "P9", "P3"],   # central, two riverside (flood-exposed), rail station
    "flood_scenarios": ["rcp8p5", "rcp4p5"],
    "flood_years": [2035, 2085],
    "flood_headline": ("RiverineInundation", "rcp8p5", 2085), #main visualization hazard x scenario x horizon combination
    "heat_scenario": "ssp585",
    "heat_year": 2050,
    "heat_hist_scenario": "historical",
    "heat_hist_year": 2005,
    "heat_base_c": 24.0,
    "electricity_tariff_eur_per_kwh": 0.155,
    "tariff_source": "Eurostat nrg_pc_205 (Spain, non-household band, incl. taxes and levies)",
    "m2_per_parking_space": 30,   # illustrative gross floor area per parking space (incl. aisles/ramps)
    "avg_vehicle_value_eur": 18000,
    "fleet_mix": {"Sedan": 0.45, "SUV": 0.25, "PickupTruck": 0.15, "Minivan": 0.10, "SportsCar": 0.05},
    "var_es_asset": "P9",                    # asset shown in the VaR / Expected Shortfall section
    "var_es_percentiles": [95, 99, 99.5],    # 0-100 percentile scale, as in the Alpha-Klima financial module
    "var_es_interpolate": True,              # few impact bins per asset, so interpolate the quantile
    "fallback_occupancy_code": -1,
    # One scoring indicator per hazard (the ECB scoring indicators).
    "score_scope": {
        "RiverineInundation": ["flood_depth"], "CoastalInundation": ["flood_depth"],
        "Wind": ["wind_speed/3s"], "Fire": ["fire_probability"], "Hail": ["hail_probability"],
        "Drought": ["cdd"], "WaterRisk": ["water_stress"], "Subsidence": ["subsidence_susceptability"],
        "Landslide": ["landslide_susceptability"], "ChronicHeat": ["cooling_degree_days/index"],
        "Snow": ["blizzard_probability"], "FreezingRain": ["freezing_rain_probability"]},
    "score_scenarios": ["historical", "rcp4p5", "rcp8p5", "ssp126", "ssp245", "ssp585"],
    "score_years": [2005, 2035, 2050, 2085],
}

# Teal-anchored, CVD-validated categorical palette (dataviz), plus ink/chrome tokens and a 0-3 risk ramp.
PALETTE = {
    "teal": "#0092A0", "orange": "#E15E0B", "violet": "#4A3AA7",
    "ink": "#0b0b0b", "ink2": "#52514e", "muted": "#898781",
    "grid": "#e1e0d9", "axis": "#c3c2b7", "surface": "#fcfcfb",
}
COMPONENT_COLOR = {"Structure": PALETTE["teal"], "Contents": PALETTE["violet"], "Vehicles": PALETTE["orange"]}
BAND_COLORS = ["#2f9e44", "#f59f00", "#e8590c", "#c92a2a"]   # score 0 no risk -> 3 high
SCORE_LABELS = {0: "No risk", 1: "Low", 2: "Medium", 3: "High"}
HAZARD_LABEL = {
    "RiverineInundation": "Riverine flood", "CoastalInundation": "Coastal flood", "Wind": "Wind",
    "Fire": "Wildfire", "Hail": "Hail", "Drought": "Drought", "WaterRisk": "Water stress",
    "Subsidence": "Subsidence", "Landslide": "Landslide", "ChronicHeat": "Chronic heat",
    "Snow": "Snow", "FreezingRain": "Freezing rain"}

OUT = Path("demo_outputs_parkings_multihazard")
OUT.mkdir(exist_ok=True)

### Helpers

One request wrapper (`assess`, POST to `/api/get_asset_impact_clean`) and small extractors for the damage ratio (`ratio_by_id`), the 0 to 3 scores (`score_matrix`), the loss-exceedance curve (`exceedance`), the impact distribution (`distribution`) and the cooling-degree-day field (`heat_cdd_at_points`). Value-at-Risk and Expected Shortfall come from `tail_metrics.var_es`, a temporary client-side module that derives them from the impact distribution the API already returns. Everything else is computed by the platform: the notebook needs only the API, no local vulnerability data.

In [ ]:
def api_post(path, payload, retries=5, timeout=240):
    "POST JSON to the API with retry/backoff on 5xx; raises on 4xx and returns the parsed body."
    last = None
    for attempt in range(retries):
        r = requests.post(API_BASE + path, json=payload, headers=HEADERS, timeout=timeout)
        if r.status_code < 500:
            r.raise_for_status()
            return r.json()
        last = r
        time.sleep(4 * (attempt + 1))
    last.raise_for_status()


def assess(items, scope, scenarios, years, calc_details=False, interpolate=True):
    "POST /api/get_asset_impact_clean for the given engine items and hazard scope."
    req = {"assets": {"items": items},
           "include_asset_level": True, "include_measures": True,
           "include_calc_details": calc_details, "clean_response": True,
           "years": years, "scenarios": scenarios,
           "calc_settings": {"hazard_scope_by_indicator": scope, "interpolate_years": interpolate}}
    return api_post("/api/get_asset_impact_clean", req)


def base_item(it):
    "Minimal engine asset (id, name, country, lat/lon) shared by every routing variant."
    return {"id": str(it["id"]), "asset_name": it["name"], "country": it.get("country", "ES"),
            "latitude": it["latitude"], "longitude": it["longitude"]}


def _match(im, hazard, scenario, year):
    "True if an impact record is for the given hazard/scenario/year key."
    k = im["key"]
    return k["hazard_type"] == hazard and k["scenario_id"] == scenario and str(k["year"]) == str(year)


def ratio_by_id(resp, hazard, scenario, year):
    "Map asset_id -> damage ratio (impact_mean) for one hazard/scenario/year, 0.0 when absent."
    out = {}
    for a in resp.get("asset_impacts", []):
        for im in a.get("impacts", []):
            if _match(im, hazard, scenario, year):
                out[a["asset_id"]] = im.get("impact_mean") or 0.0
    return out


def exceedance(resp, asset_id, hazard, scenario, year):
    "Loss-ratio exceedance curve (values, annual exceedance probabilities) for one asset."
    for a in resp.get("asset_impacts", []):
        if a["asset_id"] != str(asset_id):
            continue
        for im in a.get("impacts", []):
            if _match(im, hazard, scenario, year):
                ex = im.get("impact_exceedance") or {}
                return ex.get("values"), ex.get("exceed_probabilities")
    return None, None


def distribution(resp, asset_id, hazard, scenario, year):
    "Impact distribution (bin edges, bin probabilities) for one asset: the PMF behind the tail metrics."
    for a in resp.get("asset_impacts", []):
        if a["asset_id"] != str(asset_id):
            continue
        for im in a.get("impacts", []):
            if _match(im, hazard, scenario, year):
                d = im.get("impact_distribution") or {}
                return d.get("bin_edges"), d.get("probabilities")
    return None, None


def _measure_order(m):
    "Sort measures by horizon then scenario, so ties report the earliest scenario/year."
    key = m["key"]
    try:
        year = int(key.get("year"))
    except (TypeError, ValueError):
        year = 0
    return (year, str(key.get("scenario_id")))


def score_matrix(resp, items):
    "Worst 0-3 score per asset and hazard across the requested scenarios and horizons."
    # Returns (scores, drivers); `drivers` records the 'scenario / year' at which each worst score is
    # first reached, so every cell traces back to a specific scenario and horizon.
    ids = [str(it["id"]) for it in items]
    n = len(ids)
    best, driver = {}, {}
    for m in sorted(resp.get("risk_measures", {}).get("measures_for_assets", []), key=_measure_order):
        key = m["key"]
        hz = key["hazard_type"]
        tag = f'{key.get("scenario_id")} / {key.get("year")}'
        sc = m.get("scores") or []
        b = best.setdefault(hz, [np.nan] * n)
        d = driver.setdefault(hz, [""] * n)
        for i in range(min(n, len(sc))):
            if sc[i] is None:
                continue
            v = int(sc[i])
            if np.isnan(b[i]) or v > b[i]:
                b[i], d[i] = v, tag
    scores, drivers = pd.DataFrame({"id": ids}), pd.DataFrame({"id": ids})
    for hz in best:
        scores[hz], drivers[hz] = best[hz], driver[hz]
    return scores, drivers


def cdd_at(curve, base):
    "Cooling-degree-day value from an intensity curve at the base temperature closest to `base`."
    idx = curve.get("index_values") or []
    val = curve.get("intensities") or []
    if not idx:
        return np.nan
    j = int(np.argmin(np.abs(np.asarray(idx, float) - base)))
    return float(val[j]) if j < len(val) else np.nan


def heat_cdd_at_points(lats, lons, scenario, year, base):
    "CDD at each point for one scenario/year via /api/get_hazard_data (cooling_degree_days/index)."
    resp = api_post("/api/get_hazard_data", {"items": [{
        "request_item_id": "heat", "event_type": "ChronicHeat",
        "indicator_id": "cooling_degree_days/index", "scenario": scenario, "year": year,
        "latitudes": list(lats), "longitudes": list(lons)}]})
    return [cdd_at(c, base) for c in resp["items"][0]["intensity_curve_set"]]

In [ ]:
def style_plot(fig, title, subtitle="", height=460):
    "Apply the shared demo theme (palette, fonts, layout) and title/subtitle to a Plotly figure."
    t = "<b>" + title + "</b>"
    if subtitle:
        t += "<br><span style='font-size:12px;color:" + PALETTE["ink2"] + "'>" + subtitle + "</span>"
    fig.update_layout(
        title=dict(text=t, font=dict(size=18, color=PALETTE["ink"])),
        font=dict(family="system-ui, -apple-system, Segoe UI, sans-serif", size=13, color=PALETTE["ink2"]),
        paper_bgcolor=PALETTE["surface"], plot_bgcolor=PALETTE["surface"], height=height,
        margin=dict(t=78, b=54, l=68, r=28),
        legend=dict(orientation="h", yanchor="bottom", y=-0.24, x=0, font=dict(color=PALETTE["ink2"])))
    fig.update_xaxes(gridcolor=PALETTE["grid"], zeroline=False, linecolor=PALETTE["axis"],
                     tickfont=dict(color=PALETTE["muted"]), title_font=dict(color=PALETTE["ink2"]))
    fig.update_yaxes(gridcolor=PALETTE["grid"], zeroline=False, linecolor=PALETTE["axis"],
                     tickfont=dict(color=PALETTE["muted"]), title_font=dict(color=PALETTE["ink2"]))
    return fig


def save_plot(fig, name):
    "Write the figure to demo_outputs as standalone HTML (CDN Plotly) and return it."
    fig.write_html(OUT / (name + ".html"), include_plotlyjs="cdn")
    return fig


def save_table(df, name):
    "Write the DataFrame to demo_outputs as CSV and return it."
    df.to_csv(OUT / (name + ".csv"), index=False)
    return df

## 2. Portfolio overview

Four Seville car parks with mixed profiles (a central underground park, a riverside park, the rail station multi-storey and a western park-and-ride). Each carries the OED occupancy code 1113 ("Commercial, Parking"), a structure value, a garage contents value, and a vehicle exposure derived from `parking_spaces x occupancy x avg_vehicle_value`. Values and attributes are illustrative.

In [ ]:
all_parks = json.load(open(CONFIG["parkings_portfolio"], encoding="utf-8"))["items"]
items = [it for it in all_parks if it["id"] in CONFIG["parking_ids"]]
for it in items:
    it["group"] = "Parking"
    it["vehicle_exposure"] = round(it["parking_spaces"] * it["occupancy"] * CONFIG["avg_vehicle_value_eur"])

overview = pd.DataFrame([{"id": it["id"], "name": it["name"], "type": it["display_type"],
                          "structure_value_M": round(it["value"] / 1e6, 2),
                          "contents_value_k": round(it["contents_value"] / 1e3, 0),
                          "vehicle_exposure_M": round(it["vehicle_exposure"] / 1e6, 2)} for it in items])
save_table(overview, "t_portfolio")
tot = sum(it["value"] + it["contents_value"] + it["vehicle_exposure"] for it in items)
print(len(items), "parkings | total exposed value EUR", round(tot / 1e6, 1), "M")
display(overview)

In [ ]:
fig = gmp.show_portfolio(
    latitudes=[it["latitude"] for it in items], longitudes=[it["longitude"] for it in items],
    asset_ids=[it["id"] for it in items], names=[it["name"] for it in items],
    values=[it["value"] for it in items])
fig.update_layout(map=dict(zoom=11), height=520, margin=dict(l=0, r=0, t=46, b=0),
                  title=dict(text="<b>Seville parking portfolio</b>  (marker size = structure value)",
                             font=dict(size=16, color=PALETTE["ink"]), x=0.01))
save_plot(fig, "p_portfolio_map")
fig.show()

## 3. Complete hazard risk-score table

Every hazard the platform models is scored 0 (no risk) to 3 (high). The scores come from the `get_asset_impact` response (`risk_measures`), computed across the full set of climate scenarios (`score_scenarios`) and horizons (`score_years`, 2005 to 2085) in a single request under the parking occupancy routing (OED 1113).

There are two levels of aggregation:

- **Per-hazard score**: the worst (maximum) 0 to 3 band that hazard reaches across all requested scenarios and horizons.
- **`worst_score`**: the worst per-hazard score across all hazards for that asset, i.e. its single headline risk level. The table also reports **`worst_hazard`** and **`worst_scenario_horizon`**, so it is clear which hazard and which scenario/year drive that headline.

This is exposure, not loss.

In [ ]:
def score_item(it):
    "Asset under the parking occupancy routing (occ 1113, Europe) used for the 0-3 score table."
    item = base_item(it)
    item.update({"occupancy_code": 1113, "location": "Europe"})
    return item


resp = assess([score_item(it) for it in items], CONFIG["score_scope"],
              CONFIG["score_scenarios"], CONFIG["score_years"])
scores, drivers = score_matrix(resp, items)
scores.insert(1, "name", [it["name"] for it in items])
drivers.insert(1, "name", [it["name"] for it in items])

haz_cols = [h for h in CONFIG["score_scope"] if h in scores.columns]
scores["worst_score"] = scores[haz_cols].max(axis=1)
scores["worst_hazard"] = scores[haz_cols].idxmax(axis=1)
scores["worst_scenario_horizon"] = [drivers.loc[i, hz] for i, hz in scores["worst_hazard"].items()]
save_table(scores, "t_risk_scores")
save_table(drivers, "t_risk_score_drivers")

print("Scenarios scanned:", ", ".join(CONFIG["score_scenarios"]))
print("Horizons scanned :", ", ".join(str(y) for y in CONFIG["score_years"]))
display(scores)

Each score above is the **worst** band reached over those scenarios and horizons. The table below records *where* each one occurs, as `scenario / year`, so every cell traces back to a specific scenario and horizon (the matrix hover shows the same). Where a hazard reaches the same worst band under several scenarios or horizons, the **earliest** one is reported: `historical / 2005` therefore means the hazard is already at that band today, not only under a future scenario.

In [ ]:
display(drivers)

In [ ]:
haz_cols = [h for h in CONFIG["score_scope"] if h in scores.columns]
order = scores.sort_values("worst_score", ascending=True)
z = order[haz_cols].to_numpy(dtype=float)
cd = drivers.set_index("id").loc[order["id"], haz_cols].to_numpy()   # driving scenario / year per cell
n = len(BAND_COLORS)
colorscale = []
for i, c in enumerate(BAND_COLORS):
    colorscale += [[i / n, c], [(i + 1) / n, c]]
fig = go.Figure(go.Heatmap(
    z=z, x=[HAZARD_LABEL[h] for h in haz_cols], y=order["name"], customdata=cd,
    text=[[SCORE_LABELS.get(int(v), "") if v == v else "" for v in row] for row in z],
    texttemplate="%{text}", textfont=dict(size=11, color="white"),
    colorscale=colorscale, zmin=-0.5, zmax=3.5,
    colorbar=dict(title="Risk", tickvals=[0, 1, 2, 3], ticktext=["0 No risk", "1 Low", "2 Medium", "3 High"]),
    hovertemplate="%{y}<br>%{x}: score %{z}<br>first reached at %{customdata}<extra></extra>", xgap=2, ygap=2))
style_plot(fig, "Hazard risk-score matrix",
           f"Worst 0-3 score across {len(CONFIG['score_scenarios'])} scenarios and horizons "
           f"{min(CONFIG['score_years'])} to {max(CONFIG['score_years'])}; "
           f"hover shows the driving scenario and year", height=360)
fig.update_xaxes(tickangle=-40)
save_plot(fig, "p_score_matrix")
fig.show()

**Reading the table.** Two kinds of score sit behind these bands:

- **Specific damage function** (an impact model applied to the asset): riverine and coastal flood (Hazus garage curve), chronic heat (cooling model), wind and wildfire (generic building curve), and landslide.
- **Hazard-intensity score** (no damage curve; read straight from the hazard field against fixed thresholds): water stress, subsidence, drought, hail, snow and freezing rain.

The two material hazards for this estate, **flood** and **heat**, are priced in the next sections; the traceability section at the end records the model behind every hazard.

## 4. Expected flood loss (material)

Flood is priced by the platform in **one request** carrying two kinds of asset per parking, so each value component gets its own damage function:

- **Structure**: occupancy 1113, which routes to the Hazus COM10 garage structure curve.
- **Vehicles**: one `TransportationAsset` per fleet type (`type=Vehicle/*`, Martinez-Gomariz curves). The response returns an impact distribution per vehicle family, from which the fleet loss is reconstructed using the fleet mix and average vehicle value.

Each component's loss is its value base times the platform damage ratio, and each has its own loss-exceedance curve (used later for VaR/ES). Headline scenario RCP 8.5 at 2085.

> **Garage contents is not priced yet.** The OED occupancy mapping applies the COM10 *structure* curve only, so the platform returns no contents damage ratio for a parking. Support for a weighted structure-and-contents occupancy curve is on the way. Until then the contents value is shown as exposure in the portfolio table but excluded from the loss figures.

**Why vehicles are in the analysis.** In a car park the vehicles present are often the largest and most vulnerable flood-exposed value: a car caught in a flood is typically a total loss, while the concrete structure is far more resilient. Leaving them out would materially understate flood risk. Vehicle exposure is the occupied spaces (`parking_spaces x occupancy`) times an average vehicle value, split across a representative fleet mix.

In [ ]:
FLOOD_SCOPE = {"RiverineInundation": ["flood_depth"], "CoastalInundation": ["flood_depth"]}
HZ, SC, YR = CONFIG["flood_headline"]
OCC = CONFIG["fallback_occupancy_code"]


def parking_struct_item(it):
    "Structure asset for flood pricing: occ 1113 -> Hazus COM10, with Hazus flood defaults."
    item = base_item(it)
    item.update({"occupancy_code": 1113, "location": it.get("location", "Europe"),
                 "number_of_storeys": it.get("number_of_storeys", -1),
                 "basement": it.get("basement", 0),
                 "first_floor_height": it.get("first_floor_height", 0.305)})
    return item

# proxy vehicles portfolio (per location) 
def vehicle_items(it):
    "One TransportationAsset per fleet family (type=Vehicle/*) for USACE vehicle curves."
    out = []
    for vt in CONFIG["fleet_mix"]:
        v = base_item(it)
        v["id"] = f'{it["id"]}-veh-{vt}'
        v["asset_name"] = f'{it["name"]} [{vt}]'
        v.update({"asset_class": "TransportationAsset", "type": f"Vehicle/{vt}", "occupancy_code": OCC})
        out.append(v)
    return out

#flood risk analysis for the proxy vehicle portfolio
flood_items = [parking_struct_item(it) for it in items] + [v for it in items for v in vehicle_items(it)]
flood = assess(flood_items, FLOOD_SCOPE, CONFIG["flood_scenarios"], CONFIG["flood_years"])
ratios = ratio_by_id(flood, HZ, SC, YR)

rows = []
for it in items:
    rid = str(it["id"]); r = ratios.get(rid, 0.0)
    expected_vehicle_loss = sum(it["vehicle_exposure"] * frac * ratios.get(f"{rid}-veh-{vt}", 0.0)
                       for vt, frac in CONFIG["fleet_mix"].items())
    rows.append({"id": rid, "name": it["name"], "structure_value": it["value"],
                 "damage_ratio": r, "expected_structure_loss": it["value"] * r, "expected_vehicle_loss": expected_vehicle_loss})
flood_df = pd.DataFrame(rows)
flood_df["expected_total_flood_loss"] = flood_df[["expected_structure_loss", "expected_vehicle_loss"]].sum(axis=1)
save_table(flood_df, "t_flood_loss")
print("Total expected flood loss EUR", round(flood_df["expected_total_flood_loss"].sum() / 1e3, 1), "k/yr (", SC, YR, ")")
display(flood_df[["name", "damage_ratio", "expected_structure_loss", "expected_vehicle_loss", "expected_total_flood_loss"]])

In [ ]:
comp = flood_df.sort_values("expected_total_flood_loss", ascending=False)
fig = go.Figure()
for col, name in [("expected_structure_loss", "Structure"), ("expected_vehicle_loss", "Vehicles")]:
    fig.add_trace(go.Bar(x=comp["name"], y=comp[col] / 1e3, name=name, marker_color=COMPONENT_COLOR[name],
                         marker_line=dict(width=2, color=PALETTE["surface"]),
                         hovertemplate="%{x} " + name + ": %{y:,.1f}k EUR<extra></extra>"))
fig.update_layout(barmode="stack")
fig.update_yaxes(title="Expected flood loss (EUR k/yr)")
ct = comp[["expected_structure_loss", "expected_vehicle_loss"]].sum()
style_plot(fig, "Parking expected flood loss by component",
           f"{SC} at {YR}. Portfolio: structure {ct['expected_structure_loss']/1e3:,.0f}k, "
           f"vehicles {ct['expected_vehicle_loss']/1e3:,.0f}k EUR (contents not priced yet)", height=460)
save_plot(fig, "p_flood_components")
fig.show()

## 5. Chronic heat (material)

Heat is reported as the **additional** cooling-energy cost due to warming: the cooling demand at the scenario minus the historical baseline (excess over baseline), not the total cooling bill. Cooling-degree days are read at each location for historical 2005 and for SSP5 8.5 at 2050 (base 24 C); the extra degree-days are converted to electricity via a floor-area-scaled heat-transfer coefficient and a cooling COP, then priced with a Spain electricity tariff.

This uses `auxiliary_functions/cooling_model.py`, a small client-side calculation that previews a proposed change to the platform cooling model (`AKCoolingModel`): add the historical-baseline subtraction and per-square-metre floor-area scaling that the current platform model lacks. The 0 to 3 chronic-heat score in the table above still comes from the platform's current (absolute) model, so score and cost use different cooling formulations until that change lands.

In [ ]:
HEAT_SC, HEAT_YR = CONFIG["heat_scenario"], CONFIG["heat_year"]
lats = [it["latitude"] for it in items]; lons = [it["longitude"] for it in items]
cdd_now = heat_cdd_at_points(lats, lons, HEAT_SC, HEAT_YR, CONFIG["heat_base_c"])
cdd_hist = heat_cdd_at_points(lats, lons, CONFIG["heat_hist_scenario"], CONFIG["heat_hist_year"], CONFIG["heat_base_c"])

rows = []
for it, h, c in zip(items, cdd_hist, cdd_now):
    area = it["parking_spaces"] * CONFIG["m2_per_parking_space"]
    rows.append({"id": str(it["id"]), "name": it["name"], "floor_area_m2": area,
                 "cdd_hist": h, "cdd_2050": c,
                 "excess_cooling_cost_eur": cm.excess_cooling_cost_eur(
                     c, h, area, tariff_eur_per_kwh=CONFIG["electricity_tariff_eur_per_kwh"])})
heat_df = pd.DataFrame(rows)
heat_df["cdd_change"] = (heat_df["cdd_2050"] - heat_df["cdd_hist"]).clip(lower=0)
save_table(heat_df, "t_heat")
print("Total additional cooling cost vs baseline EUR",
      round(heat_df["excess_cooling_cost_eur"].sum() / 1e3, 1), "k/yr",
      "| mean CDD change:", round(np.nanmean(heat_df["cdd_change"])))
display(heat_df)

In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(x=heat_df["name"], y=heat_df["cdd_hist"], name="Historical (2005)",
                     marker_color=PALETTE["muted"], marker_line_width=0,
                     hovertemplate="%{x} historical: %{y:,.0f} CDD<extra></extra>"))
fig.add_trace(go.Bar(x=heat_df["name"], y=heat_df["cdd_2050"], name=f"{HEAT_SC} ({HEAT_YR})",
                     marker_color=PALETTE["orange"], marker_line_width=0,
                     hovertemplate="%{x} " + str(HEAT_YR) + ": %{y:,.0f} CDD<extra></extra>"))
fig.update_layout(barmode="group")
fig.update_yaxes(title="Cooling degree days (base 24 C)")
style_plot(fig, "Cooling demand: historical vs mid-century", "Per parking", height=440)
save_plot(fig, "p_cdd_change")
fig.show()

fig2 = go.Figure(go.Bar(x=heat_df["name"], y=heat_df["excess_cooling_cost_eur"] / 1e3, marker_color=PALETTE["teal"],
                        marker_line_width=0, text=[f"{v/1e3:,.1f}k" for v in heat_df["excess_cooling_cost_eur"]],
                        textposition="outside", textfont=dict(color=PALETTE["ink2"]),
                        hovertemplate="%{x}: %{y:,.1f}k EUR/yr<extra></extra>"))
fig2.update_yaxes(title="Additional cooling cost (EUR k/yr)")
style_plot(fig2, "Additional cooling cost vs baseline, by parking",
           f"Excess over historical 2005; {HEAT_SC} at {HEAT_YR}, tariff "
           f"{CONFIG['electricity_tariff_eur_per_kwh']} EUR/kWh", height=430)
save_plot(fig2, "p_cooling_cost")
fig2.show()

## 6. Financial roll-up

The two material hazards combine into an annual climate cost: expected flood loss (structure and vehicles) plus the additional cooling-energy cost vs baseline.

In [ ]:
fin = flood_df.merge(heat_df[["id", "excess_cooling_cost_eur"]], on="id", how="left")
roll = pd.DataFrame({
    "component": ["Flood: structure", "Flood: vehicles", "Heat: additional cooling"],
    "eur_per_year": [fin["expected_structure_loss"].sum(), fin["expected_vehicle_loss"].sum(),
                     np.nansum(fin["excess_cooling_cost_eur"])]})
save_table(roll, "t_financial_rollup")
print("Total expected annual climate cost EUR", round(roll["eur_per_year"].sum() / 1e3, 1), "k/yr")
fig = go.Figure(go.Bar(x=roll["component"], y=roll["eur_per_year"] / 1e3,
                       marker_color=[PALETTE["teal"], PALETTE["orange"], PALETTE["ink2"]],
                       marker_line_width=0, text=[f"{v/1e3:,.1f}k" for v in roll["eur_per_year"]],
                       textposition="outside", textfont=dict(color=PALETTE["ink2"]),
                       hovertemplate="%{x}: %{y:,.1f}k EUR/yr<extra></extra>"))
fig.update_yaxes(title="Expected annual cost (EUR k/yr)")
style_plot(fig, "Annual climate cost by component", "Expected flood loss + cooling energy", height=440)
save_plot(fig, "p_financial_rollup")
fig.show()
display(roll)

### Flood tail risk: Value-at-Risk and Expected Shortfall

Expected annual loss is an average; a risk view also needs the tail. **Value-at-Risk (VaR)** at confidence c is the flood loss exceeded with annual probability `1 - c` (VaR 99% is the 1-in-100-year loss); **Expected Shortfall (ES)** is the average loss in that worst `1 - c` tail, so it is always at least the VaR.

The platform's productionised financial engine (portfolio VaR/ES) is a separate engagement and is not called here. These are derived from the per-asset impact distribution the impact endpoint already returns (`impact_distribution`), one distribution per component, since the structure and each vehicle family are sent as their own asset. All components are driven by the same flood depth, so their losses are comonotonic and both VaR and ES add; vehicles dominate the tail because a flooded car is a near-total loss. Contents is absent for the reason given in section 4.

The calculation lives in `auxiliary_functions/tail_metrics.py`, a **temporary client-side module that will be replaced by the platform API**. Its math mirrors Alpha-Klima's financial module: VaR is the discrete quantile of the loss distribution, the smallest loss whose cumulative probability reaches the confidence level, and Expected Shortfall is the mean of the tail beyond it. These numbers will not move once the midway endpoints land. Percentiles follow a 0 to 100 convention.

Each asset carries only a handful of impact bins, so `CONFIG["var_es_interpolate"]` is on: the quantile is read off a piecewise-linear CDF that assumes uniform loss density within each bin, which is the assumption the platform's own impact bins are built on. Turn it off for the strict discrete quantile, which snaps to a bin centre and is the better choice for aggregated portfolios with rich distributions.

The table breaks the tail down by component; the two charts (rendered with the shared `financial_metrics_plots` helpers) show the total flood VaR and Expected Shortfall by percentile and climate scenario.

In [ ]:
VE_ID = CONFIG["var_es_asset"]
PCTS = CONFIG["var_es_percentiles"]
INTERP = CONFIG["var_es_interpolate"]
ve = next(it for it in items if str(it["id"]) == VE_ID)


def component_var_es(scenario, year):
    "VaR/ES per component (structure, vehicles) for one scenario/year, as {percentile: (VaR, ES)}."
    be, bp = distribution(flood, VE_ID, HZ, scenario, year)
    struct = tm.var_es(be, bp, PCTS, scale=ve["value"], interpolate=INTERP)

    # Every vehicle family sees the same flood depth, so family losses are comonotonic: VaR and ES add.
    veh = {p: (0.0, 0.0) for p in PCTS}
    for vt, frac in CONFIG["fleet_mix"].items():
        be, bp = distribution(flood, f"{VE_ID}-veh-{vt}", HZ, scenario, year)
        fam = tm.var_es(be, bp, PCTS, scale=ve["vehicle_exposure"] * frac, interpolate=INTERP)
        veh = {p: (veh[p][0] + fam[p][0], veh[p][1] + fam[p][1]) for p in PCTS}
    return {"structure": struct, "vehicles": veh}


# Per-component table for the headline scenario.
comp = component_var_es(SC, YR)
ve_rows = []
for p in PCTS:
    tot = tuple(sum(comp[k][p][j] for k in comp) for j in (0, 1))
    ve_rows.append({"confidence": f"{p / 100:.1%}", "return_period": f"1-in-{round(1 / (1 - p / 100))}",
                    "VaR_structure_k": round(comp["structure"][p][0] / 1e3),
                    "VaR_vehicles_k": round(comp["vehicles"][p][0] / 1e3),
                    "VaR_total_k": round(tot[0] / 1e3), "ES_total_k": round(tot[1] / 1e3)})
var_es_df = pd.DataFrame(ve_rows)
save_table(var_es_df, "t_var_es")
print(ve["name"], "| 1-in-100 flood VaR EUR", var_es_df.loc[var_es_df["return_period"] == "1-in-100", "VaR_total_k"].iat[0],
      "k | ES EUR", var_es_df.loc[var_es_df["return_period"] == "1-in-100", "ES_total_k"].iat[0], "k")
display(var_es_df)

In [ ]:
# Total (comonotonic) VaR and ES per scenario and percentile, plotted with the shared helpers.
totals = {s: component_var_es(s, YR) for s in CONFIG["flood_scenarios"]}
combined = {s: {p: tuple(sum(cv[k][p][j] for k in cv) for j in (0, 1)) for p in PCTS}
            for s, cv in totals.items()}

# One payload carries both metrics; each plotter selects its own rows by label.
resp = tm.metrics_response(combined, term="long")
fig = fmp.plot_var(resp, name=ve["name"] + " flood")
save_plot(fig, "p_var")
fig.show()
fig = fmp.plot_es(resp, name=ve["name"] + " flood")
save_plot(fig, "p_es")
fig.show()

## 7. Per-asset provenance

One row per parking: what was sent to the engine for the priced hazards, the value bases, the flood damage ratio and the three flood components, plus the cooling cost.

In [ ]:
prov = flood_df[["id", "name", "structure_value", "damage_ratio", "expected_structure_loss",
                 "expected_vehicle_loss", "expected_total_flood_loss"]].copy()
prov = prov.merge(heat_df[["id", "cdd_2050", "excess_cooling_cost_eur"]], on="id", how="left")
prov = prov.merge(scores[["id", "worst_score"]], on="id", how="left")
prov["flood_routing"] = "occ 1113 (COM10 structure) + Vehicle/* (occ -1)"
save_table(prov, "t_provenance")
display(prov.sort_values("expected_total_flood_loss", ascending=False).reset_index(drop=True))

## 8. Traceability and methodology

The model behind every hazard, and why it was chosen. `IMPACT` scores come from a vulnerability damage ratio; `HAZARD` scores are read from the hazard intensity against fixed thresholds. Every flood loss is computed by the platform; the only client-side step is the chronic-heat cost. Garage contents is not priced yet (the COM10 occupancy mapping returns structure only).

Full documentation of the hazard models, vulnerability curves and scoring methodology is on the Alpha-Klima docs site: **https://platform.alpha-klima.com/docs/index.html**. The table below summarises, per hazard, what this specific run used.

In [ ]:
TRACE = [
 ("RiverineInundation", "flood_depth", "occ 1113 -> Hazus COM10", "WRI flood depth + Hazus COM10 garage structure (543)", "IMPACT", "platform", "Specific garage curve for parking occupancy", "WRI Aqueduct; FEMA Hazus"),
 ("CoastalInundation", "flood_depth", "occ 1113 -> Hazus COM10", "WRI coastal flood depth + Hazus COM10 (543)", "IMPACT", "platform", "Same garage curve as riverine", "WRI Aqueduct; FEMA Hazus"),
 ("Flood vehicles", "flood_depth", "TransportationAsset Vehicle/* (occ -1)", "Martinez-Gomariz urban-flood vehicle curves", "IMPACT", "platform", "One asset per vehicle family; exposure = spaces x occupancy x avg value x fleet mix", "Martinez-Gomariz et al. 2019"),
 ("Flood contents", "flood_depth", "not available", "COM10 contents curve (357), not exposed", "IMPACT", "not priced", "The occupancy mapping applies the COM10 structure curve only; a weighted structure-and-contents curve is on the way", "FEMA Hazus"),
 ("Flood tail (VaR/ES)", "flood_depth", "per-component impact_distribution", "Discrete quantile + tail mean over the impact PMF; interpolated within bins when CONFIG['var_es_interpolate']", "IMPACT", "client-side", "Mirrors the Alpha-Klima financial module; temporary until the platform exposes tail metrics through the API", "Alpha-Klima financial module"),
 ("ChronicHeat", "cooling_degree_days/index", "occ 1113 -> RealEstate", "Score: platform cooling model (absolute). Cost: excess CDD (scenario - historical 2005) x UA(floor area x per-m2 coeff) / COP", "IMPACT", "platform score / client-side cost", "Cost is the excess over baseline scaled by building size; previews a proposed AKCoolingModel change", "ECB SPS No.48; cooling model"),
 ("Wind", "wind_speed/3s", "occ 1113", "WISC 3s gust + generic building wind curve", "IMPACT", "platform", "COM10 has no wind curve, so a generic building curve applies under occ 1113", "WISC; JRC/Huizinga"),
 ("Fire", "fire_probability", "occ 1113", "Generic wildfire damage curve", "IMPACT", "platform", "Generic curve applies under occ 1113; no parking-specific fire curve", "Alpha-Klima wildfire"),
 ("Landslide", "landslide_susceptability", "occ 1113", "Landslide susceptibility damage curve", "IMPACT", "platform", "Susceptibility-based damage", "JRC"),
 ("WaterRisk", "water_stress", "occ 1113", "None (score from intensity)", "HAZARD", "platform", "Score-only: no damage curve, read from water-stress ratio", "WRI Aqueduct 4.0"),
 ("Subsidence", "subsidence_susceptability", "occ 1113", "None (score from intensity)", "HAZARD", "platform", "Score-only: susceptibility index thresholds", "JRC"),
 ("Drought", "cdd", "occ 1113", "None (score from intensity)", "HAZARD", "platform", "Score-only: consecutive dry days thresholds", "Copernicus"),
 ("Hail", "hail_probability", "occ 1113", "None (score from intensity)", "HAZARD", "platform", "Score-only: annual large-hail probability", "TU Delft RAIN"),
 ("Snow", "blizzard_probability", "occ 1113", "None (score from intensity)", "HAZARD", "platform", "Score-only: annual blizzard probability", "TU Delft RAIN"),
 ("FreezingRain", "freezing_rain_probability", "occ 1113", "None (score from intensity)", "HAZARD", "platform", "Score-only: freezing-rain probability", "TU Delft RAIN"),
]
trace_df = pd.DataFrame(TRACE, columns=["hazard", "indicator_id", "routing", "model / curve resolved",
                                        "score source", "computed by", "why this model", "reference"])
save_table(trace_df, "t_traceability")
display(trace_df)

### Sources and assumptions

- Flood structure uses the Hazus COM10 (garage) curve reached via OED occupancy 1113. Vehicles use the Martinez-Gomariz 2019 urban-flood curves, one asset per vehicle family; vehicle exposure is `parking_spaces x occupancy x avg_vehicle_value` (about 18,000 EUR) split over an illustrative fleet mix. Both components, including their loss-exceedance curves, are computed by the platform: the notebook needs no local vulnerability data.
- Garage contents is not priced. The COM10 occupancy mapping applies the structure curve only, so the platform returns no contents damage ratio; a weighted structure-and-contents occupancy curve is on the way. Contents value is retained in the portfolio as exposure but excluded from the loss figures.
- Value-at-Risk and Expected Shortfall are computed client-side by `auxiliary_functions/tail_metrics.py` from the per-asset `impact_distribution` the API already returns, and rendered with the shared `financial_metrics_plots` helpers. The math mirrors Alpha-Klima's financial module, so the figures will not move when the platform exposes these metrics through the API. That module is temporary and should be deleted once those endpoints land.
- Chronic heat is reported as the additional cooling-energy cost vs the historical 2005 baseline, scaled by building floor area (about 30 m2 per parking space) and priced at a Spain electricity tariff of about 0.155 EUR/kWh. Tariff source: Eurostat `nrg_pc_205` (Spain, non-household band, incl. taxes and levies), recorded in `CONFIG["tariff_source"]`. Costs are in **constant (real) terms**: the current tariff is applied to future cooling demand with no energy-price inflation, so figures across horizons are directly comparable. This client-side calculation (`auxiliary_functions/cooling_model.py`) previews a proposed change to the platform cooling model (baseline subtraction + per-square-metre floor-area scaling); the 0 to 3 heat score still uses the platform's current absolute model.
- The risk-score table is built from a single request under the parking occupancy routing (OED 1113), which resolves all hazards; score-only hazards carry no damage curve.
- Full model documentation (hazard models, vulnerability curves and the scoring methodology) is on the Alpha-Klima docs site: https://platform.alpha-klima.com/docs/index.html
- Asset values and parking attributes are illustrative. Results are for screening, not regulatory submission.